In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/clean_nav_data.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["fund_name", "date"])

In [2]:
df["daily_return"] = df.groupby("fund_name")["nav"].pct_change()
df = df.dropna()

In [3]:
volatility = df.groupby("fund_name")["daily_return"].std() * np.sqrt(252)
print(volatility)

fund_name
Axis_Bluechip      26.347177
HDFC_Top100_NAV     0.152110
ICICI_Bluechip           NaN
Kotak_Bluechip      0.154357
Nippon_LargeCap     0.167082
SBI_Bluechip        0.087204
Name: daily_return, dtype: float64


In [4]:
cagr = df.groupby("fund_name").apply(
    lambda x: (x["nav"].iloc[-1] / x["nav"].iloc[0]) ** (252 / len(x)) - 1
)

print(cagr)

fund_name
Axis_Bluechip      0.480703
HDFC_Top100_NAV    0.247753
ICICI_Bluechip     0.157757
Kotak_Bluechip     0.172971
Nippon_LargeCap    0.155347
SBI_Bluechip       0.001153
dtype: float64


In [5]:
sharpe = df.groupby("fund_name")["daily_return"].mean() / df.groupby("fund_name")["daily_return"].std()
sharpe = sharpe * np.sqrt(252)

print(sharpe)

fund_name
Axis_Bluechip      0.268550
HDFC_Top100_NAV    1.531517
ICICI_Bluechip          NaN
Kotak_Bluechip     1.110762
Nippon_LargeCap    0.947025
SBI_Bluechip       0.058397
Name: daily_return, dtype: float64


In [6]:
metrics = pd.DataFrame({
    "CAGR": cagr,
    "Volatility": volatility,
    "Sharpe Ratio": sharpe
})

print(metrics)

                     CAGR  Volatility  Sharpe Ratio
fund_name                                          
Axis_Bluechip    0.480703   26.347177      0.268550
HDFC_Top100_NAV  0.247753    0.152110      1.531517
ICICI_Bluechip   0.157757         NaN           NaN
Kotak_Bluechip   0.172971    0.154357      1.110762
Nippon_LargeCap  0.155347    0.167082      0.947025
SBI_Bluechip     0.001153    0.087204      0.058397


In [7]:
metrics.to_csv("../data/processed/performance_metrics.csv")
print("Saved successfully")

Saved successfully


In [1]:
with open("../sql/queries.sql", "r") as file:
    sql_script = file.read()

sql_script

'Query 1\n\nSELECT fund_name, AVG(nav) AS avg_nav\nFROM fact_nav\nGROUP BY fund_name\nORDER BY avg_nav DESC;\n\nQuery 2\n\nSELECT fund_name, COUNT(*) AS total_records\nFROM fact_nav\nGROUP BY fund_name;\n\nQuery 3\n\nSELECT *\nFROM fact_nav\nWHERE nav = (SELECT MAX(nav) FROM fact_nav);\n\n\nQuery 4\n\nSELECT *\nFROM fact_nav\nWHERE nav = (SELECT MIN(nav) FROM fact_nav);\n\n\nQuery 5\n\nSELECT fund_name,\n       MAX(nav) - MIN(nav) AS nav_range\nFROM fact_nav\nGROUP BY fund_name;\n\n\nQuery 6\n\nSELECT fund_name, AVG(nav) AS avg_nav\nFROM fact_nav\nGROUP BY fund_name;\n\nQuery 7\n\nSELECT fund_name,\n       MAX(nav) - MIN(nav) AS volatility_proxy\nFROM fact_nav\nGROUP BY fund_name\nORDER BY volatility_proxy DESC;\n\n\nQuery 8\n\nSELECT date, COUNT(*) \nFROM fact_nav\nGROUP BY date;\n\n\nQuery 9\n\nSELECT *\nFROM fact_nav f1\nWHERE date = (\n    SELECT MAX(date)\n    FROM fact_nav f2\n    WHERE f1.fund_name = f2.fund_name\n);\n\nQuery 10\n\nSELECT DISTINCT fund_name\nFROM fact_nav\nWHERE

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/db/bluestock_mf.db")

KeyboardInterrupt: 

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/db/bluestock_mf.db")

In [4]:
import pandas as pd
import sqlite3

In [5]:
conn = sqlite3.connect("../data/db/bluestock_mf.db")
pd.read_sql("SELECT COUNT(*) FROM fact_nav", conn)


,COUNT(*)
0,19786


In [6]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,nav_history
1,fact_nav
2,fact_performance


In [7]:
pd.read_sql("SELECT * FROM fact_nav LIMIT 5", conn)

,date,nav,fund_name,daily_return
0,2013-01-02 00:00:00.000000,24.0036,Axis_Bluechip,0.000238
1,2013-01-03 00:00:00.000000,24.0092,Axis_Bluechip,0.000233
2,2013-01-04 00:00:00.000000,24.0147,Axis_Bluechip,0.000229
3,2013-01-06 00:00:00.000000,24.0258,Axis_Bluechip,0.000462
4,2013-01-07 00:00:00.000000,24.0314,Axis_Bluechip,0.000233


In [8]:
pd.read_sql("""
SELECT fund_name, AVG(nav) AS avg_nav
FROM fact_nav
GROUP BY fund_name
""", conn)

,fund_name,avg_nav
0,Axis_Bluechip,3392.862319
1,HDFC_Top100_NAV,89.988964
2,ICICI_Bluechip,56.718266
3,Kotak_Bluechip,101.957386
4,Nippon_LargeCap,46.107706
5,SBI_Bluechip,125.844526


In [9]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,nav_history
1,fact_nav
2,fact_performance


In [10]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,nav_history
1,fact_nav
2,fact_performance


In [11]:
pd.read_sql("SELECT * FROM fact_nav LIMIT 5", conn)

,date,nav,fund_name,daily_return
0,2013-01-02 00:00:00.000000,24.0036,Axis_Bluechip,0.000238
1,2013-01-03 00:00:00.000000,24.0092,Axis_Bluechip,0.000233
2,2013-01-04 00:00:00.000000,24.0147,Axis_Bluechip,0.000229
3,2013-01-06 00:00:00.000000,24.0258,Axis_Bluechip,0.000462
4,2013-01-07 00:00:00.000000,24.0314,Axis_Bluechip,0.000233


In [12]:
pd.read_sql("SELECT * FROM fact_performance", conn)

,CAGR,Volatility,Sharpe Ratio
0,0.001558,26.339781,0.266198
1,0.000874,0.152149,1.122975
2,0.000633,0.154310,0.722478
3,0.000571,0.167037,0.588789
4,0.000005,0.087177,-0.628918


In [13]:
pd.read_sql("""
SELECT fund_name, AVG(nav) AS avg_nav
FROM fact_nav
GROUP BY fund_name
""", conn)

,fund_name,avg_nav
0,Axis_Bluechip,3392.862319
1,HDFC_Top100_NAV,89.988964
2,ICICI_Bluechip,56.718266
3,Kotak_Bluechip,101.957386
4,Nippon_LargeCap,46.107706
5,SBI_Bluechip,125.844526
